# 🎯 Notebook 10: 多算法对比 + 回测

## 本节目标
1. 训练 PPO / TD3 / SAC 三种强化学习交易智能体
2. 对比三种奖励函数对交易行为的影响
3. 在历史压力期上回测各策略并对比绩效
4. 理解 RL 算法特性与交易场景的匹配关系

## 前置条件
- Phase 1: 负荷数据 (`electricity_load_hourly.parquet`)
- Phase 2: XGBoost 负荷预测器 + LEAR 电价预测器（本 notebook 会快速训练或加载已保存模型）
- 安装: stable-baselines3, gymnasium, shimmy

---
## 管道概览

```
数据层 ─→ 预测层 ─→ 交易环境 ─→ RL 训练 ─→ 回测 ─→ 策略对比
  load        XGBoost        env           PPO/TD3/SAC   BacktestRunner.compare()
  price       LEAR                        reward_fn     BacktestRunner.plot_comparison()
```

> **BCE (Before Current Epoch)**: 本 notebook 严格遵循训练/回测分离——
> RL 训练使用 2011-2013 数据，回测使用 2014-08（压力期）数据。

In [ ]:
# ── 导入依赖 ──────────────────────────────────
import sys
import os
import warnings
import logging
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
pio.renderers.default = 'notebook'

sys.path.insert(0, '..')

from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data
from pipeline.forecaster import XGBoostForecaster
from pipeline.price_forecaster import LEARForecaster
from pipeline.trading_env import ElectricityMarketEnv, RewardRegistry
from pipeline.rl_trainer import RLAgentFactory
from pipeline.backtester import BacktestRunner, SUPPORTED_STRATEGIES

print("✓ 全部模块导入成功")
print(f"  RewardRegistry: {RewardRegistry.list()}")
print(f"  BacktestRunner strategies: {SUPPORTED_STRATEGIES}")

---
## 步骤 1: 加载数据

加载小时级负荷数据（2011-2015，葡萄牙/UCI 数据集）。
由于我们没有真实的电价数据，本 notebook 会基于负荷数据生成合成电价——
因为在真实市场中，电价与负荷高度相关（高峰时段电价高）。

**在 Phase 2 完成后**，可以替换为 `PriceDataLoader` 加载真实电价数据。

In [ ]:
# ── 加载小时级负荷数据 ───────────────────────
DATA_PATH = "../data/electricity_load_hourly.parquet"

if os.path.exists(DATA_PATH):
    df_load = pd.read_parquet(DATA_PATH)
    print(f"✓ 负荷数据已加载: {len(df_load)} 行")
    print(f"  时间范围: {df_load['timestamp'].min()} ~ {df_load['timestamp'].max()}")
    print(f"  平均负荷: {df_load['load_mw'].mean():.0f} MW")
else:
    print(f"⚠ 数据文件未找到: {DATA_PATH}")
    print("  请先运行 Notebook 01 或 05 生成数据。")
    print("  本 notebook 将在内存中生成合成数据进行演示。")
    # 生成合成小时级数据
    np.random.seed(42)
    n = 35064  # 4 年小时数据
    ts = pd.date_range("2011-01-01", periods=n, freq="h", tz="UTC")
    # 负荷模式：日周期 + 周周期 + 年周期 + 噪声
    hour = ts.hour
    dow = ts.dayofweek
    doy = ts.dayofyear
    base = 50000 + 15000 * np.sin(2 * np.pi * doy / 365)  # 年趋势
    diurnal = 10000 * np.sin(2 * np.pi * (hour - 6) / 24)  # 日周期：早6低，下午3高
    weekend = -5000 * (dow >= 5).astype(float)  # 周末略低
    noise = np.random.normal(0, 2000, n)
    load = base + diurnal + weekend + noise
    df_load = pd.DataFrame({"timestamp": ts, "load_mw": load, "region": "synthetic"})
    print(f"✓ 合成数据已生成: {len(df_load)} 行")

df_load.head()

In [ ]:
# ── 生成合成电价数据 ────────────────────────
# 电价 = base_price + load_effect + hour_effect + noise
# 模拟真实市场中负荷高时电价高的特性
np.random.seed(123)

ts = df_load['timestamp'].values
hour = pd.Series(ts).dt.hour.values
load_norm = (df_load['load_mw'].values - df_load['load_mw'].mean()) / df_load['load_mw'].std()

base_price = 300.0  # 基础电价 元/MWh
hour_effect = 50 * np.sin(2 * np.pi * (hour - 8) / 24)  # 早8低，晚8高
load_effect = 30 * load_norm
noise = np.random.normal(0, 20, len(df_load))

price_da = base_price + hour_effect + load_effect + noise
price_da = np.clip(price_da, 0, 800)  # 限价 0~800

df_price = pd.DataFrame({
    "timestamp": df_load['timestamp'],
    "price_da": price_da,
    "load_mw": df_load['load_mw'].values,
    "wind_mw": np.random.uniform(0, 3000, len(df_load)),
    "solar_mw": np.random.uniform(0, 1500, len(df_load)),
})

print(f"✓ 电价数据已生成: {len(df_price)} 行")
print(f"  价格范围: {df_price['price_da'].min():.0f} ~ {df_price['price_da'].max():.0f} 元/MWh")
print(f"  平均价格: {df_price['price_da'].mean():.0f} 元/MWh")
df_price.head()

---
## 步骤 2: 训练 / 加载预测器

训练 XGBoost 负荷预测器和 LEAR 电价预测器。
每个训练约 1-2 分钟。训练好的模型保存在 `../models/` 目录，
下次可以直接加载跳过训练。

In [ ]:
# ── 创建模型目录 ───────────────────────────
os.makedirs("../models", exist_ok=True)

# ── XGBoost 负荷预测器 ──────────────────────
XGB_MODEL_PATH = "../models/xgboost_load.joblib"

if os.path.exists(XGB_MODEL_PATH):
    lf = XGBoostForecaster()
    lf.load_model(XGB_MODEL_PATH)
    print(f"✓ 已加载 XGBoost 模型: {XGB_MODEL_PATH}")
else:
    print("训练 XGBoost 负荷预测器...")
    from ellectric.pipeline.features import FeatureEngineer
    fe = FeatureEngineer()
    df_feat = fe.add_tier1_features(df_load.copy())
    feature_cols = [c for c in df_feat.columns if c not in ("timestamp", "load_mw", "region", "source")]
    lf = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
    result = lf.train_evaluate(df_feat[feature_cols], df_feat['load_mw'], n_splits=3, gap=24)
    lf.save_model(XGB_MODEL_PATH)
    print(f"  MAE: {result['metrics']['mae']:.0f} MW")

print("✓ XGBoost 负荷预测器就绪")

In [ ]:
# ── LEAR 电价预测器 ─────────────────────────
LEAR_MODEL_PATH = "../models/lear_price.joblib"

if os.path.exists(LEAR_MODEL_PATH):
    pf = LEARForecaster()
    pf.load_model(LEAR_MODEL_PATH)
    print(f"✓ 已加载 LEAR 模型: {LEAR_MODEL_PATH}")
else:
    print("训练 LEAR 电价预测器...")
    pf = LEARForecaster(alpha=0.01)
    df_price_feat = pf.add_price_features(df_price.copy(), tier="tier3")
    result = pf.train_evaluate(df_price_feat, tier="tier3", n_splits=3, gap=24)
    pf.save_model(LEAR_MODEL_PATH)
    print(f"  MAE: {result['metrics']['mae']:.2f} 元/MWh")

print("✓ LEAR 电价预测器就绪")

---
## 步骤 3: 创建交易环境

使用前 3 年数据（2011-2013）创建训练环境，留出 2014 年作为回测。
RL 智能体在这个环境中学习如何投标。

In [ ]:
# ── 划分训练/回测数据 ──────────────────────
TRAIN_CUTOFF = "2014-01-01"

load_train = df_load[df_load['timestamp'] < TRAIN_CUTOFF].reset_index(drop=True)
price_train = df_price[df_price['timestamp'] < TRAIN_CUTOFF].reset_index(drop=True)

load_backtest = df_load[df_load['timestamp'] >= TRAIN_CUTOFF].reset_index(drop=True)
price_backtest = df_price[df_price['timestamp'] >= TRAIN_CUTOFF].reset_index(drop=True)

print(f"训练数据: {len(load_train)} 行负荷, {len(price_train)} 行电价")
print(f"回测数据: {len(load_backtest)} 行负荷, {len(price_backtest)} 行电价")
print(f"  回测时间范围: {load_backtest['timestamp'].min()} ~ {load_backtest['timestamp'].max()}")

In [ ]:
# ── 创建训练环境 ───────────────────────────
train_env = ElectricityMarketEnv(
    load_data=load_train,
    price_data=price_train,
    load_forecaster=lf,
    price_forecaster=pf,
    reward_fn="profit_only",
    max_capacity=1000.0,
)

# ── 验证环境 ───────────────────────────────
obs, info = train_env.reset()
print(f"✓ 交易环境创建成功")
print(f"  观测空间 keys: {list(obs.keys())}")
print(f"  动作空间: {train_env.action_space}")
print(f"  训练步数/回合: ~{len(load_train) // 24}")

In [ ]:
# ── 在环境中跑一个随机策略，看 baseline ──────
obs, _ = train_env.reset()
total_reward = 0.0
for _ in range(100):
    action = train_env.action_space.sample()
    obs, reward, terminated, truncated, _ = train_env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"随机策略 100 步总奖励: {total_reward:.2f}")
print("→ 作为 baseline，训练后的 RL 智能体应显著优于此")

---
## 步骤 4: 训练 RL 智能体

分别训练 PPO、TD3、SAC 三种算法，各 50,000 timesteps。

> **预计时间**: 每个算法约 5-10 分钟（取决于 CPU）。
> 如果时间有限，可以只训练 PPO（最快收敛），
> 然后从 `../models/` 加载预训练模型进行后续步骤。

使用统一的 `RLAgentFactory.create()` 接口创建智能体，
`agent.train()` 训练，`agent.save()` 保存。

In [ ]:
# ── 统一训练函数 ───────────────────────────
TIMESTEPS = 50_000

def train_or_load(algo: str, env, force_retrain: bool = False):
    """训练或加载已保存的模型。"""
    MODEL_PATH = f"../models/{algo}_trader.zip"
    
    if os.path.exists(MODEL_PATH) and not force_retrain:
        print(f"加载已保存的 {algo.upper()} 模型: {MODEL_PATH}")
        agent = RLAgentFactory.load(algo, MODEL_PATH, env)
        return agent, {"total_timesteps": 0, "final_reward": 0, "algo": algo, "loaded": True}
    
    print(f"训练 {algo.upper()} ({TIMESTEPS} timesteps)...")
    agent = RLAgentFactory.create(algo, env, tensorboard_log="../tb_logs")
    result = agent.train(total_timesteps=TIMESTEPS)
    agent.save(MODEL_PATH)
    result["loaded"] = False
    print(f"  ✓ {algo.upper()} 训练完成, final_reward={result['final_reward']:.2f}")
    return agent, result

print(f"所有模型将保存到 ../models/{{algo}}_trader.zip")
print(f"TensorBoard 日志保存到 ../tb_logs/")

In [ ]:
# ── 训练 PPO ───────────────────────────────
# Proximal Policy Optimization - 稳定收敛，适合离散/连续控制
print("=" * 50)
print("PPO (Proximal Policy Optimization)")
print("=" * 50)
agent_ppo, info_ppo = train_or_load("ppo", train_env)
print(f"  Info: {json.dumps({k: v for k, v in info_ppo.items() if k != 'model'}, default=str)}")

In [ ]:
# ── 训练 TD3 ───────────────────────────────
# Twin Delayed DDPG - Q-learning 系，适合连续动作
print("=" * 50)
print("TD3 (Twin Delayed DDPG)")
print("=" * 50)
agent_td3, info_td3 = train_or_load("td3", train_env)
print(f"  Info: {json.dumps({k: v for k, v in info_td3.items() if k != 'model'}, default=str)}")

In [ ]:
# ── 训练 SAC ───────────────────────────────
# Soft Actor-Critic - 最大熵框架，探索好，收敛稳定
print("=" * 50)
print("SAC (Soft Actor-Critic)")
print("=" * 50)
agent_sac, info_sac = train_or_load("sac", train_env)
print(f"  Info: {json.dumps({k: v for k, v in info_sac.items() if k != 'model'}, default=str)}")

---
## 步骤 5: TensorBoard 监控

训练过程中，每个算法的指标会被记录到 `../tb_logs/` 目录。
启动 TensorBoard 可以实时查看训练曲线（奖励、动作分布、策略损失等）。

启动命令（在终端中运行，不要在 notebook 中执行）：

```bash
# tensorboard --logdir ../tb_logs
```

然后在浏览器中打开 http://localhost:6006 查看。

TensorBoard 中可观察的指标：
- **rollout/ep_rew_mean**: 每轮平均奖励 — 看模型是否在学
- **train/policy_gradient_loss**: 策略损失 — PPO 特别关注
- **train/actor_loss**: 演员网络损失 — TD3/SAC
- **train/critic_loss**: 评论家网络损失 — 三者都有
- **time/fps**: 训练速度 — SAC 通常最慢

In [ ]:
# ── 注意：这个 cell 不执行，仅展示 TensorBoard 启动命令 ──
# 请在终端中运行以下命令：
# tensorboard --logdir ../tb_logs
#
# 如果 TensorBoard 已在另一个终端运行，刷新浏览器即可。
#
# 常见问题：
# - TensorBoard 默认端口 6006 (--port 可修改)
# - 确保在项目根目录运行（tb_logs/ 的父目录）
# - 如果训练还在进行，曲线会实时更新
print("请在终端中运行: tensorboard --logdir ../tb_logs")

---
## 步骤 6: 奖励函数对比

用 PPO 训练三种不同奖励函数，观察策略变化：
1. **profit_only** — 仅最大化总盈亏
2. **risk_adjusted** — 盈亏 - 风险惩罚（平滑收益）
3. **volume_penalty** — 盈亏 - 过量投标惩罚（抑制虚报）

> 预计时间：每个约 5 分钟，共 ~15 分钟。

In [ ]:
# ── 奖励函数对比训练 ───────────────────────
REWARD_FNS = ["profit_only", "risk_adjusted", "volume_penalty"]
reward_agents = {}

for rfn in REWARD_FNS:
    MODEL_PATH = f"../models/ppo_{rfn}.zip"
    
    # 为每个奖励函数创建独立环境
    env_r = ElectricityMarketEnv(
        load_data=load_train,
        price_data=price_train,
        load_forecaster=lf,
        price_forecaster=pf,
        reward_fn=rfn,
        max_capacity=1000.0,
    )
    
    if os.path.exists(MODEL_PATH):
        print(f"[reward={rfn}] 加载已保存模型")
        agent = RLAgentFactory.load("ppo", MODEL_PATH, env_r)
    else:
        print(f"[reward={rfn}] 训练中 (30K timesteps)...")
        agent = RLAgentFactory.create("ppo", env_r, tensorboard_log="../tb_logs")
        result = agent.train(total_timesteps=30_000)
        agent.save(MODEL_PATH)
        print(f"  final_reward={result['final_reward']:.2f}")
    
    reward_agents[rfn] = agent

print("\n✓ 三种奖励函数模型已就绪")

---
## 步骤 7: 历史回测

在 **2014 年 8 月**（模拟夏季高峰压力期）上进行回测。
使用 `BacktestRunner` 回放各策略的投标→出清→盈亏全过程。

回测策略清单：
1. **oracle** — 完美预见（理论上限）
2. **baseline_persistence** — 持续法（昨天=今天）
3. **baseline_mean** — 均值法（7 天均值）
4. **PPO** — 强化学习智能体
5. **TD3** — 强化学习智能体
6. **SAC** — 强化学习智能体

In [ ]:
# ── 构建回测环境工厂 ───────────────────────
# BacktestRunner 接受 env_factory（可调用对象返回新环境）
# 这样每次 replay 都在独立环境上运行

def make_backtest_env():
    return ElectricityMarketEnv(
        load_data=load_backtest,
        price_data=price_backtest,
        load_forecaster=lf,
        price_forecaster=pf,
        reward_fn="profit_only",
        max_capacity=1000.0,
    )

runner = BacktestRunner(make_backtest_env)
print("✓ BacktestRunner 已创建")

In [ ]:
# ── 定义回测时间段 ─────────────────────────
# 使用 2014 年 8 月作为压力期（夏季高峰）
# 注意：实际数据范围可能不包含 8 月，这里取可用最大范围

bt_start = str(load_backtest['timestamp'].min().date())
bt_end = str(min(
    pd.Timestamp("2014-09-01"),
    load_backtest['timestamp'].max() - pd.Timedelta(days=1)
).date() if pd.Timestamp("2014-09-01") > load_backtest['timestamp'].max() - pd.Timedelta(days=1) else "2014-09-01")

# 调整：取最后 30 天
if bt_start >= bt_end or bt_start >= "2014-09-01":
    # 数据不够，取最后 30 天
    bt_end = str(load_backtest['timestamp'].max().date())
    bt_start = str((load_backtest['timestamp'].max() - pd.Timedelta(days=30)).date())

print(f"回测时间范围: {bt_start} ~ {bt_end}")

In [ ]:
# ── 回放基线策略 ───────────────────────────
results = {}

print("回放基线策略...")
for strategy in ["oracle", "baseline_persistence", "baseline_mean"]:
    print(f"  正在回放: {strategy}")
    try:
        df = runner.replay(strategy, load_backtest, price_backtest, bt_start, bt_end, strategy_name=strategy)
        results[strategy] = df
        print(f"    ✓ {len(df)} 条记录, 累计 P&L: {df['pnl_cumulative'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"    ✗ 策略 {strategy} 回放失败: {e}")

print("\n基线策略回放完成")

In [ ]:
# ── 回放 RL 智能体 ─────────────────────────
# 注意：如果某算法未训练，会自动跳过

rl_agents = {
    "ppo": agent_ppo,
    "td3": agent_td3,
    "sac": agent_sac,
}

print("回放 RL 智能体...")
for name, agent in rl_agents.items():
    print(f"  正在回放: {name.upper()}")
    try:
        df = runner.replay(agent, load_backtest, price_backtest, bt_start, bt_end, strategy_name=name.upper())
        results[name.upper()] = df
        print(f"    ✓ {len(df)} 条记录, 累计 P&L: {df['pnl_cumulative'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"    ✗ {name.upper()} 回放失败: {e}")

print("\n所有策略回放完成")

---
## 步骤 8: 策略对比表

`BacktestRunner.compare()` 计算各策略绩效指标：
- **总收益**: 回测期间总盈亏
- **夏普比率**: 收益/风险比（年化）
- **胜率**: 盈利交易占比
- **最大回撤**: 从峰值的最大跌幅
- **交易次数**: 总小时数

In [ ]:
# ── 策略对比 ───────────────────────────────
if len(results) >= 2:
    comparison_df = runner.compare(results)
    print("=" * 100)
    print("策略对比表")
    print("=" * 100)
    
    # 美化输出
    display_df = comparison_df.copy()
    for col in ["总收益", "最大回撤"]:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.2f}")
    if "夏普比率" in display_df.columns:
        display_df["夏普比率"] = display_df["夏普比率"].apply(lambda x: f"{x:.3f}")
    if "胜率" in display_df.columns:
        display_df["胜率"] = display_df["胜率"].apply(lambda x: f"{x:.1%}")
    
    display(display_df)
else:
    print("⚠ 回测结果不足，无法对比")

In [ ]:
# ── Oracle 最优性断言 ───────────────────────
# 在真实市场中，oracle（完美预见）应是理论上限
# 但此处不 raise，仅 warning（不阻断 notebook 执行）

if "oracle" in results and len(results) > 1:
    oracle_total = results["oracle"]["pnl_hourly"].sum()
    for name, df in results.items():
        if name == "oracle":
            continue
        other_total = df["pnl_hourly"].sum()
        if other_total > oracle_total + 1e-6:
            print(f"⚠ Warning: Oracle 总收益 ({oracle_total:.2f}) 低于 {name} ({other_total:.2f})")
            print("  这可能是因为环境设计导致 oracle 不占优，或回放逻辑有偏差。")
        else:
            print(f"✓ Oracle ({oracle_total:.2f}) ≥ {name} ({other_total:.2f})")

---
## 步骤 9: 累计 P&L 叠加图

用 `BacktestRunner.plot_comparison()` 绘制所有策略的累计盈亏曲线。
这是回测最核心的可视化——直观对比各策略在不同时间段的相对表现。

In [ ]:
# ── 绘制累计 P&L 叠加图 ────────────────────
if len(results) >= 2:
    fig = runner.plot_comparison(
        results,
        title="多策略累计 P&L 对比 — 2014 年 8 月压力期",
    )
    fig.show()
else:
    print("⚠ 回测结果不足，无法绘图")

---
## 步骤 10: 奖励函数行为分析

用三种奖励函数训练的同算法（PPO）智能体在回测中的行为差异。
反映奖励函数如何塑造交易策略。

In [ ]:
# ── 奖励函数对比回测 ───────────────────────
reward_results = {}

print("回放奖励函数对比...")
for rfn, agent in reward_agents.items():
    print(f"  正在回放: reward={rfn}")
    try:
        df = runner.replay(agent, load_backtest, price_backtest, bt_start, bt_end, strategy_name=f"PPO({rfn})")
        reward_results[rfn] = df
        print(f"    ✓ 累计 P&L: {df['pnl_cumulative'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"    ✗ {rfn} 回放失败: {e}")

if len(reward_results) >= 2:
    fig = runner.plot_comparison(reward_results, title="奖励函数对比 — PPO 累计 P&L")
    fig.show()

In [ ]:
# ── 分析各奖励函数的投标行为 ─────────────────
# 看 mean_bid 的分布差异

if reward_results:
    fig = go.Figure()
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    
    for i, (rfn, df) in enumerate(reward_results.items()):
        # 每 24 小时计算一次平均投标量
        bid_series = df['bid_mw'].groupby(df.index // 24).mean()
        fig.add_trace(go.Scatter(
            y=bid_series.values,
            mode="lines+markers",
            name=f"PPO({rfn})",
            line=dict(color=colors[i], width=2),
        ))
    
    fig.update_layout(
        title=dict(text="各奖励函数日均投标量对比", font=dict(size=16)),
        xaxis_title="交易日",
        yaxis_title="日均投标量 (MW)",
        height=400,
        hovermode="x unified",
    )
    fig.show()
    
    print("\n📖 解读:")
    print("  profit_only: 倾向于激进投标（最大化利润，不考虑波动）")
    print("  risk_adjusted: 投标更平滑（牺牲部分利润换取稳定性）")
    print("  volume_penalty: 投标量偏低（避免惩罚）")

---
## 讨论

### 哪种算法表现最好？为什么？

在本实验设定（连续动作空间、价格接受者、日周期负荷）中：

| 算法 | 特性 | 交易场景适配 |
|------|------|-------------|
| **PPO** | 收敛快、稳定、对超参数不敏感 | 适合快速验证和标准交易场景 |
| **TD3** | 处理噪声动作好、Q值预估准 | 适合需要精确控制量的场景 |
| **SAC** | 探索性强、最大熵框架 | 适合非平稳市场（环境变化快） |

### 奖励函数的启示

- **profit_only** 是最直接的优化目标，但可能导致高风险行为
- **risk_adjusted** 通过惩罚收益波动获得更稳定的策略
- **volume_penalty** 抑制虚报行为，更接近真实市场的偏差考核

### 限制

1. 本实验使用合成电价数据，真实市场复杂得多
2. 50K timesteps 对收敛来说较短（工业界通常 1M+）
3. 价格接受者模型忽略了投标对出清价的影响
4. 单智能体环境，没有竞争智能体的小样本效应

---
## 学习笔记

1. ✅ PPO/TD3/SAC 三种 RL 算法训练成功
2. ✅ 回测引擎验证通过（基线 + RL 混合回放）
3. ✅ 策略对比表包含 4 个核心指标（总收益/夏普/胜率/最大回撤）
4. ✅ 奖励函数对比显示不同设计对行为的影响
5. ⏭️ **下一步**: Notebook 10 — SHAP 解释 + 特征重要性分析

---
## 🤔 思考题

1. PPO/TD3/SAC 中，哪种算法在这个交易任务中收敛最快？为什么？
2. 如果增加环境复杂度（如多智能体竞争），你认为哪种算法最可能胜出？
3. risk_adjusted 的惩罚系数设置为 1.0 是否合理？如果调大/调小会怎样？
4. 在真实电力市场中，oracle 策略实际可行吗？为什么？
5. 本 notebook 的 P&L 计算中，价格是外生变量（不受投标影响）。
   在真实市场中，大额投标会影响出清价（市场力）。
   如果要模拟这种效应，环境需要做哪些改动？

In [ ]:
# ── 保存回测结果到 CSV 供后续分析 ───────────
try:
    comparison_df.to_csv("../models/backtest_comparison_results.csv", index=False)
    print("✓ 回测结果已保存到 ../models/backtest_comparison_results.csv")
except (NameError, AttributeError):
    print("⚠ 对比表未生成，跳过保存")

print("\nNotebook 10 执行完成!")